<a href="https://colab.research.google.com/github/nupriya/Gernerative-AI-and-Agentic-AI/blob/main/qlora_updated_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun Apr  5 17:53:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
#!pip install sympy==1.12

In [ ]:
# Install dependencies
import sys
!{sys.executable} -m pip install -q transformers datasets peft accelerate bitsandbytes

# Imports
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

# =========================
# 1. MODEL NAME
# =========================
# NOTE: Requires HF access (Meta approval)
model_name = "meta-llama/Llama-2-7b-hf"
# Alternative (no approval needed):
# model_name = "NousResearch/Llama-2-7b-hf"

# =========================
# 2. QUANTIZATION CONFIG (QLoRA)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# =========================
# 3. LOAD MODEL
# =========================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# =========================
# 4. TOKENIZER FIX
# =========================
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # IMPORTANT

# =========================
# 5. MODEL SETTINGS
# =========================
model.config.use_cache = False
model.gradient_checkpointing_enable()

# =========================
# 6. LoRA CONFIG
# =========================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =========================
# 7. LOAD DATASET
# =========================
dataset = load_dataset("imdb", split="train[:1%]")

# =========================
# 8. TOKENIZATION
# =========================
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    result["labels"] = result["input_ids"].copy()
    return result

dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

# =========================
# 9. DATA COLLATOR
# =========================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# =========================
# 10. TRAINING ARGUMENTS
# =========================
training_args = TrainingArguments(
    output_dir="./qlora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    optim="paged_adamw_32bit",
    save_total_limit=2,
    report_to="none"
)

# =========================
# 11. TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

# =========================
# 12. TRAIN
# =========================
trainer.train()

# =========================
# 13. SAVE MODEL
# =========================
model.save_pretrained("qlora-model")
tokenizer.save_pretrained("qlora-model")

print("✅ Training completed and model saved!")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]